# Fairness-Aware Housing Price Prediction in Cook County

## 1. Problem and Fairness Framework

This project predicts residential sale prices in Cook County, Illinois, while auditing whether prediction errors differ systematically across neighborhood socioeconomic and demographic contexts.

The project has two objectives:

1. **Predictive performance** — estimate observed residential sale prices accurately.
2. **Fairness auditing** — compare error magnitude and direction across predefined Census-tract groups.

Property characteristics and geographic variables are used for prediction. Census tract-level demographic and socioeconomic variables from the American Community Survey (ACS) are reserved for **post-model auditing**, rather than used as direct demographic predictors.

For an observed sale price $y_i$ and prediction $\hat{y}_i$, overprediction occurs when $\hat{y}_i > y_i$ and underprediction when $\hat{y}_i < y_i$. Later notebooks evaluate MAE, RMSE, MAPE, signed percentage error, overprediction rate, and prediction ratios across groups.

In [ ]:
from pathlib import Path
import os
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ZIP = PROJECT_ROOT / 'data' / 'raw' / 'cook_county_data.zip'

if not DATA_ZIP.exists():
    raise FileNotFoundError(
        f'Missing {DATA_ZIP}. See data/README.md for setup instructions.'
    )

with zipfile.ZipFile(DATA_ZIP, 'r') as z:
    with z.open('cook_county_train.csv') as f:
        data = pd.read_csv(f)

print(f'Dataset shape: {data.shape}')
print(f'Sale-year range: {data["Sale Year"].min()}–{data["Sale Year"].max()}')
print(f'Unique Census tracts: {data["Census Tract"].nunique():,}')
data.head()

## 2. Core Variables and Geographic Linkage

The prediction target is `Sale Price`. Candidate property predictors include building and land area, bathrooms, age, property class, construction and condition variables, garage/improvement characteristics, transaction timing, and geography.

`Census Tract` provides the bridge to tract-level ACS context. Cook County uses Illinois state FIPS `17` and county FIPS `031`. Census tract identifiers are standardized to six digits and combined with the state/county codes to form an 11-digit GEOID.

The original property data contain 1,265 unique tracts. Validation against the 2019 ACS tract inventory confirmed that all 1,265 property tracts matched official Cook County tract GEOIDs.

In [ ]:
data['Census Tract Code'] = (
    pd.to_numeric(data['Census Tract'], errors='coerce')
      .astype('Int64')
      .astype('string')
      .str.zfill(6)
)

data['tract_geoid'] = '17031' + data['Census Tract Code']

ACS_YEAR = 2019
ACS_URL = f'https://api.census.gov/data/{ACS_YEAR}/acs/acs5'
CENSUS_API_KEY = os.getenv('CENSUS_API_KEY')

tract_params = {
    'get': 'NAME',
    'for': 'tract:*',
    'in': 'state:17 county:031',
}
if CENSUS_API_KEY:
    tract_params['key'] = CENSUS_API_KEY

response = requests.get(ACS_URL, params=tract_params, timeout=30)
response.raise_for_status()
tract_rows = response.json()
official_tracts = pd.DataFrame(tract_rows[1:], columns=tract_rows[0])
official_tracts['tract_geoid'] = (
    official_tracts['state'] + official_tracts['county'] + official_tracts['tract']
)

property_geoids = set(data['tract_geoid'].dropna())
official_geoids = set(official_tracts['tract_geoid'])
unmatched = property_geoids - official_geoids

print(f'Official Cook County ACS tracts: {len(official_geoids):,}')
print(f'Property tracts: {len(property_geoids):,}')
print(f'Unmatched property tracts: {len(unmatched):,}')

## 3. ACS Socioeconomic and Demographic Context

The fairness audit uses 2019 ACS 5-year tract-level estimates. The variables below describe neighborhood context, not the race, ethnicity, or income of an individual property owner or buyer.

Selected ACS measures:

- Median household income — `B19013_001E`
- Population below poverty — `B17001_002E` / poverty universe `B17001_001E`
- Total population for race/ethnicity composition — `B03002_001E`
- White non-Hispanic — `B03002_003E`
- Black non-Hispanic — `B03002_004E`
- Asian non-Hispanic — `B03002_006E`
- Hispanic/Latino of any race — `B03002_012E`

In [ ]:
acs_vars = [
    'B19013_001E',
    'B17001_001E', 'B17001_002E',
    'B03002_001E', 'B03002_003E', 'B03002_004E',
    'B03002_006E', 'B03002_012E',
]

acs_params = {
    'get': 'NAME,' + ','.join(acs_vars),
    'for': 'tract:*',
    'in': 'state:17 county:031',
}
if CENSUS_API_KEY:
    acs_params['key'] = CENSUS_API_KEY

response = requests.get(ACS_URL, params=acs_params, timeout=30)
response.raise_for_status()
rows = response.json()
acs = pd.DataFrame(rows[1:], columns=rows[0])

for col in acs_vars:
    acs[col] = pd.to_numeric(acs[col], errors='coerce')
    acs.loc[acs[col] < 0, col] = np.nan

acs['tract_geoid'] = acs['state'] + acs['county'] + acs['tract']
acs['median_household_income'] = acs['B19013_001E']
acs['poverty_rate'] = 100 * acs['B17001_002E'] / acs['B17001_001E']
acs['pct_white_non_hispanic'] = 100 * acs['B03002_003E'] / acs['B03002_001E']
acs['pct_black_non_hispanic'] = 100 * acs['B03002_004E'] / acs['B03002_001E']
acs['pct_asian_non_hispanic'] = 100 * acs['B03002_006E'] / acs['B03002_001E']
acs['pct_hispanic'] = 100 * acs['B03002_012E'] / acs['B03002_001E']

audit_cols = [
    'tract_geoid', 'median_household_income', 'poverty_rate',
    'pct_white_non_hispanic', 'pct_black_non_hispanic',
    'pct_asian_non_hispanic', 'pct_hispanic'
]

data_fair = data.merge(acs[audit_cols], on='tract_geoid', how='left', validate='many_to_one')
print(f'Merged shape: {data_fair.shape}')
data_fair[audit_cols[1:]].describe().T

## 4. Fairness Group Construction

Quartile-based groups are used instead of arbitrary fixed thresholds. This produces similarly sized comparison groups while respecting the strongly skewed distributions of several neighborhood-composition variables.

The groups are defined separately for median household income, poverty rate, and each racial/ethnic composition measure. These labels will later be attached to held-out test observations for post-model auditing.

In [ ]:
data_fair['income_group'] = pd.qcut(
    data_fair['median_household_income'], q=4,
    labels=['Q1: Lowest income', 'Q2', 'Q3', 'Q4: Highest income']
)

data_fair['poverty_group'] = pd.qcut(
    data_fair['poverty_rate'], q=4,
    labels=['Q1: Lowest poverty', 'Q2', 'Q3', 'Q4: Highest poverty']
)

composition_variables = {
    'pct_white_non_hispanic': 'white_composition_group',
    'pct_black_non_hispanic': 'black_composition_group',
    'pct_asian_non_hispanic': 'asian_composition_group',
    'pct_hispanic': 'hispanic_composition_group',
}

for variable, group_col in composition_variables.items():
    data_fair[group_col] = pd.qcut(
        data_fair[variable], q=4,
        labels=['Q1: Lowest share', 'Q2', 'Q3', 'Q4: Highest share'],
        duplicates='drop'
    )

group_cols = [
    'income_group', 'poverty_group', 'white_composition_group',
    'black_composition_group', 'asian_composition_group',
    'hispanic_composition_group'
]

for col in group_cols:
    print(f'\n{col}')
    print(data_fair[col].value_counts(dropna=False).sort_index())

## 5. Fairness Evaluation Design

The fairness audit is conducted **after** a predictive model has been trained and evaluated on a held-out test set. The audit compares both absolute error and systematic error direction across the predefined neighborhood groups.

Primary group-level metrics include:

- **MAE** — average dollar error magnitude
- **MAPE** — average percentage error magnitude
- **Overprediction rate** — proportion with predicted price above observed price
- **Mean signed percentage error** — direction and magnitude of systematic error

Later analysis also adjusts comparisons for the underlying sale-price distribution and evaluates robustness using median-based percentage errors. Group disparities are interpreted as descriptive model-audit evidence, not by themselves as evidence of causal or unlawful discrimination.

In [ ]:
def add_prediction_errors(df, actual_col, predicted_col):
    result = df.copy()
    result['error'] = result[predicted_col] - result[actual_col]
    result['absolute_error'] = result['error'].abs()
    result['percentage_error'] = result['error'] / result[actual_col]
    result['absolute_percentage_error'] = result['percentage_error'].abs()
    result['overpredicted'] = result[predicted_col] > result[actual_col]
    return result


def fairness_summary(df, group_col):
    return (
        df.groupby(group_col, observed=True)
          .agg(
              n=('error', 'size'),
              mae=('absolute_error', 'mean'),
              mape=('absolute_percentage_error', 'mean'),
              overprediction_rate=('overpredicted', 'mean'),
              mean_signed_percentage_error=('percentage_error', 'mean'),
          )
    )